In [28]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import re

from tqdm import trange, tqdm

ciph = np.load('cipher.npy')
ciph = np.clip(ciph, 0, 26)   # clamp all values to valid range 0-26

# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()


In [29]:
GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [30]:
train = ciph[0:16160, :, :]
valid = ciph[16160:18180, :, :]
test  = ciph[18180:, :, :]
print(train.shape)

(16160, 2, 50)


In [31]:
def nums_to_chars(arr):
    lookup = "abcdefghijklmnopqrstuvwxyz "
    return np.vectorize(lambda x: lookup[int(x)])(arr)

In [32]:
x_train = torch.tensor(train[:, 0, :], dtype=torch.long)   # ciphered
y_train = torch.tensor(train[:, 1, :], dtype=torch.long)   # original

x_valid = torch.tensor(valid[:, 0, :], dtype=torch.long)
y_valid = torch.tensor(valid[:, 1, :], dtype=torch.long)

x_test  = torch.tensor(test[:, 0, :],  dtype=torch.long)
y_test  = torch.tensor(test[:, 1, :],  dtype=torch.long)

In [33]:
class ShakeText(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [34]:
batch_size = 64
train_loader = DataLoader(ShakeText(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(ShakeText(x_test, y_test), batch_size=batch_size, shuffle=False)
valid_loader = DataLoader(ShakeText(x_valid, y_valid), batch_size=batch_size, shuffle=False)

In [35]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)
print(x.dtype)
print(y.dtype)

torch.Size([64, 50])
torch.Size([64, 50])
torch.int64
torch.int64


In [36]:
bidirec = False

class rnn(nn.Module) :
    def __init__ (self, vocab_size, input_size, hidden_size, output_size, pad_idx):
        super(rnn, self).__init__()
        self.embedding = nn.Embedding(vocab_size, input_size, pad_idx)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True, num_layers=1, bidirectional=bidirec)
        self.fc = nn.Linear((1+int(bidirec))*hidden_size, output_size)

    def forward (self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x) # replace underscore with a pair of underscores for LSTM
        return self.fc(out) 

In [37]:
model = rnn(27, 128, 1024, 27, 0).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
n_epochs = 50
print(model)

rnn(
  (embedding): Embedding(27, 128, padding_idx=0)
  (rnn): RNN(128, 1024, batch_first=True)
  (fc): Linear(in_features=1024, out_features=27, bias=True)
)


In [38]:
def train_epoch(model, train_loader, criterion, optimizer, loss_logger):
    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        
        optimizer.zero_grad()
        outputs = model(data.to(device))
        loss = criterion(outputs.permute(0,2,1), target.to(device).long())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        loss_logger.append(loss.item())

    return model, optimizer, loss_logger

def test_model(model, test_loader, criterion, loss_logger):
    with torch.no_grad():
        correct_predictions = 0
        total_predictions = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(test_loader, desc="Testing", leave=False)):   
            
            outputs = model(data.to(device))        
            _, predicted = torch.max(outputs, dim=-1)
            correct_predictions += (predicted == target.to(device)).sum().item()
            total_predictions += target.numel()
            loss = criterion(outputs.permute(0,2,1), target.to(device).long())
            loss_logger.append(loss.item())
            
        acc = (correct_predictions/total_predictions) * 100.0
        
        return loss_logger, acc

train_loss = []
test_loss  = []
test_acc   = []

In [39]:
for i in trange(n_epochs, desc="Epoch", leave=False):
    model, optimizer, train_loss = train_epoch(model, train_loader, criterion, optimizer, train_loss)
    
    test_loss, acc = test_model(model, test_loader, criterion, test_loss)
    test_acc.append(acc)

print("Final Accuracy: %.2f%%" % acc)

Final Accuracy: 96.62%


In [40]:
torch.save(model.state_dict(), "model.pth")